<div style="background:linear-gradient(120deg,#0d366b 0%,#2a78d6 55%,#eb6834 100%);padding:32px 28px;border-radius:14px;color:white;font-family:system-ui,-apple-system,'Segoe UI',sans-serif">
<div style="font-size:13px;letter-spacing:2px;opacity:.85">ENVEDA · CASMI 2026 · A TEST BENCH</div>
<div style="font-size:30px;font-weight:700;margin:8px 0 6px">Does your candidate score pick the right isomer?</div>
<div style="font-size:16px;opacity:.95">A leak-checked Class-2 test bench for any candidate-scoring idea, in 3 CPU minutes — plus a first measurement of the in-silico fragmentation channel on top-3 isomers.</div>
</div>

The strong public pipelines in this competition combine several evidence channels to rank candidate structures. One of them is **MetFrag-lite**: break the candidate's bonds on paper and measure how much of the observed spectrum the resulting fragment masses explain. It is a natural, principled idea, and in standalone ablations it helps.

This notebook builds a small, reusable **test bench** and asks one narrow question of it, because it is the question that decides the leaderboard:

> When the true structure is already in the top 3 candidates and something else is ranked first, **does a given score pick the truth?**

That is where the remaining score lives. In the Class-2 simulation below, the truth is at rank 1 for 74 of 250 molecules; it sits at rank 2 for 47 and rank 3 for 25. Promoting the rank-2 cases alone would be worth roughly +0.09 MRR@25.

**First result.** Six variants of the fragmentation score, tested on the 72 molecules where the truth is in the top 3 but not first: rates run from **0.18 to 0.29**, against **0.33** for random choice among three. On this sample none of them separates same-formula isomers better than chance. That does not make the channel useless in a full pipeline — it says that for *this* job, choosing among already-plausible isomers, combinatorial bond-breaking as implemented does not add discrimination. The bench is here so you can test your own idea in the same way, in one line.

Everything below runs end to end on CPU in about 3 minutes.

---

## How to read this notebook

It is written for two readers at once. **Plain-language paragraphs** explain what each step does and why it exists; boxes marked **Expert note** give the technical detail. If you are new to mass spectrometry, read the glossary at the very end first — it is short.

### The whole flow in one table

| Step | What happens | Why it is needed |
|---|---|---|
| 1. Setup | Convert formulas to exact masses; convert measured ions to neutral molecule masses | Candidates are neutral molecules; the instrument weighs charged ions |
| 2. Class-2 simulation | Pick 250 known natural products and **delete every spectrum of them from every library** | Imitates a molecule nobody has measured before — the hard case that decides the leaderboard |
| 3. Candidate pool | For each molecule, list every known structure within ±10 ppm of its mass | These same-mass structures are the "suspects"; the true one is among them ~99% of the time |
| 4. Spectral kernels | Code to compare two spectra | Used both to rank candidates and to prove the simulation is leak-free |
| 5. Ranking | Rank the suspects by **analog propagation** | Produces a top-3 list, the same way strong public pipelines do |
| 6. Fragmentation | Break each suspect's bonds on paper; count how much of the spectrum its pieces explain | This is the channel under test |
| 7. Measurement | When the truth is in the top 3 but not first, does fragmentation pick it? | The one question that matters for MRR@25 |
| 8. Conclusion | What the numbers mean and what to do about it | |

### The one number we care about

Imagine a 3-horse race where you know the winner is one of three horses. If you pick at random you are right **1 time in 3 (33%)**. A useful signal should beat that. A signal that is right *less* often than random is not merely useless — it is pointing at the wrong horse.

In [ ]:
import os, sys, glob, re, time, gc, subprocess, warnings
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import scipy.sparse as sp
import matplotlib as mpl, matplotlib.pyplot as plt
from numba import njit, prange
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
T0 = time.time()
def tick(m): print(f'[{time.time()-T0:6.0f}s] {m}', flush=True)
def note(md): display(Markdown(md))

def find(name):
    h = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return sorted(h, key=len)[0] if h else None
TRAIN, TEST = find('train.parquet'), find('test.parquet')
COCO = find('coconut_structures.parquet')
try:
    import rdkit
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q', find('rdkit-*.whl')])
    import rdkit
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

BLUE, ORANGE, AQUA, MUTED = '#2a78d6', '#eb6834', '#1baf7a', '#898781'
INK, INK2, GRID, BASE, SURF = '#0b0b0b', '#52514e', '#e1e0d9', '#c3c2b7', '#fcfcfb'
mpl.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF, 'axes.edgecolor': BASE,
    'axes.linewidth': 0.8, 'axes.labelcolor': INK2, 'axes.titlecolor': INK, 'axes.titleweight': 'semibold',
    'axes.titlesize': 12, 'axes.titlelocation': 'left', 'axes.titlepad': 10, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'axes.spines.top': False,
    'axes.spines.right': False, 'font.size': 10, 'figure.dpi': 110, 'legend.frameon': False, 'lines.linewidth': 2})
print('rdkit', rdkit.__version__)

## 1. Setup: masses, adducts, structures

**What this does.** A mass spectrometer never weighs the molecule itself. It weighs the molecule *plus* whatever charged particle attached to it during ionisation — most often a single proton (`[M+H]+`), sometimes sodium (`[M+Na]+`), or it loses a proton (`[M-H]-`). The `adduct` column records which. Subtracting that particle's mass gives the **neutral mass** — the mass of the bare molecule — and that is the only number a spectrum and a candidate structure can agree on.

We also need the exact mass of every candidate structure. Chemical formulas like `C16H20O6` are converted to a mass by summing the **monoisotopic** mass of each atom (the mass of its most common isotope; carbon is exactly 12.000, hydrogen 1.00783, oxygen 15.99491).

The cell then loads the scalar columns of `train.parquet` (2.5 million spectra, about 276,000 distinct structures) and builds an index from each structure to its spectra. Structures are identified by **InChIKey14**, the first 14 characters of a standard chemical identifier that encodes atom connectivity but ignores stereochemistry — the same key the competition metric uses.

> **Expert note.** `ADDUCT_SHIFT` holds the ten adducts that occur in the test set. Values are the monoisotopic mass of the adduct minus the electron mass where relevant (e.g. `[M+H]+` = 1.007276, the proton). Neutral mass = `precursor_mz − shift` for singly charged ions. Structures are grouped by categorical code of `inchikey14` so that all later lookups are integer indexing, and `s_mass` is the median formula mass across a structure's rows (they should all agree).

In [ ]:
MONO = {'C': 12.0, 'H': 1.00782503207, 'N': 14.0030740048, 'O': 15.99491461956, 'S': 31.972071, 'P': 30.97376163,
        'F': 18.99840322, 'Cl': 34.96885268, 'Br': 78.9183371, 'I': 126.904473, 'Na': 22.98976928, 'K': 38.96370668,
        'Si': 27.9769265, 'B': 11.0093054, 'Se': 79.9165213, 'D': 2.01410178}
TOK = re.compile(r'([A-Z][a-z]?)(\d*)')
def fmass(f):
    if not isinstance(f, str) or not f or TOK.sub('', f): return np.nan
    m = 0.0
    for el, k in TOK.findall(f):
        if el not in MONO: return np.nan
        m += MONO[el] * (int(k) if k else 1)
    return m
PROTON = 1.007276
ADDUCT_SHIFT = {'[M+H]+': PROTON, '[M+NH4]+': 18.033823, '[M-H2O+H]+': PROTON - 18.010565, '[M-2H2O+H]+': PROTON - 36.021129,
                '[M+Na]+': 22.989218, '[M+K]+': 38.963158, '[M-H]-': -PROTON, '[M-H2O-H]-': -PROTON - 18.010565,
                '[M+CH2O2-H]-': 44.998201, '[M+Cl]-': 34.969402}
def neutral_mass(prec, adduct):
    return np.asarray(prec, float) - pd.Series(np.asarray(adduct, dtype=object)).map(ADDUCT_SHIFT).astype(float).to_numpy()

pf = pq.ParquetFile(TRAIN)
tr = pf.read(columns=['normalized_smiles', 'inchikey14', 'molecular_formula', 'ingest_lib', 'adduct', 'ionization_mode', 'precursor_mz']).to_pandas()
N = len(tr)
lib_arr = tr.ingest_lib.astype(str).to_numpy()
ikcat = tr.inchikey14.astype('category'); ik = ikcat.cat.codes.to_numpy().astype(np.int64); S = int(ik.max()) + 1
s_key = np.asarray(ikcat.cat.categories.astype(str))
fcat = tr.molecular_formula.astype('category'); fm = pd.Series(fcat.cat.categories).map(fmass).to_numpy()
row_mass = np.where(fcat.cat.codes.to_numpy() >= 0, fm[fcat.cat.codes.to_numpy()], np.nan)
g = pd.DataFrame({'ik': ik, 'smi': tr.normalized_smiles.to_numpy(), 'm': row_mass}).groupby('ik')
s_smiles = g.smi.first().reindex(range(S)).to_numpy(); s_mass = g.m.median().reindex(range(S)).to_numpy(); del g
row_order = np.argsort(ik, kind='stable'); iks = ik[row_order]
s_start = np.searchsorted(iks, np.arange(S)); s_end = np.searchsorted(iks, np.arange(S), side='right')
mode_arr = (tr.ionization_mode.astype(str).to_numpy() == 'positive').astype(np.int8)
prec_arr = tr.precursor_mz.to_numpy().astype(np.float64)
IS_NP_EX = lib_arr == 'enveda-np-examples'; IS_E180 = lib_arr == 'enveda-180'
tick(f'train {N:,} spectra / {S:,} structures')

## 2. A Class-2 simulation that is actually held out

**What this does.** The competition's hidden test molecules come in three kinds. **Class 1** molecules have reference spectra somewhere in the training library, so they can be found by simply matching spectra. **Class 2** molecules are known structures (they exist in chemical databases) but *nobody has ever recorded a spectrum of them* — they can only be found by reasoning about structure. **Class 3** are entirely new molecules. Class 2 is where most of the achievable score lives, so that is what we simulate.

To imitate a Class-2 molecule we take a real, known molecule and pretend its spectra do not exist: we delete every spectrum of it from the library, then try to identify it anyway. The 250 `enveda-np-examples` structures are ideal for this — they are natural products measured on the same instrument (timsTOF) as the hidden test.

**Why this step is dangerous.** Those 250 molecules were also measured by other laboratories. They appear in riken, gnps, mona, massbank, pluskal_ms2 and msdial too — tens of thousands of spectra. If we only removed the `enveda-np-examples` rows, the molecule would still be sitting in the library under another name, a plain spectral search would find it, and every result downstream would look far better than it really is. This exact mistake has already cost teams in this competition real submissions.

So the cell removes by **structure identity (InChIKey14) across every library**, and Section 4 then *proves* the removal worked by running the library search and demanding zero self-matches.

> **Expert note.** `LIB_OK` is a boolean row mask over the full training table; the analog index, the library search and the candidate pool all consult it, so no channel can see a held-out structure. The verification in Section 4 is an `assert`: if a self-match appeared, the notebook would stop rather than report inflated numbers.

In [ ]:
VAL = np.unique(ik[IS_NP_EX])                       # the 250 held-out structures
in_val = np.isin(ik, VAL)
print(f'held-out structures: {len(VAL)}')
print(f'their spectra inside enveda-np-examples : {int((IS_NP_EX).sum()):,}')
print(f'their spectra in OTHER libraries        : {int((in_val & ~IS_NP_EX).sum()):,}   <- these must go too')
LIB_OK = ~in_val                                    # the Class-2 library: no spectrum of any held-out structure
ls_order = np.argsort(s_mass); ls_order = ls_order[np.isfinite(s_mass[ls_order])]; ls_sorted = s_mass[ls_order]
def lib_structs(M, da=0.01):
    return ls_order[np.searchsorted(ls_sorted, M - da):np.searchsorted(ls_sorted, M + da, side='right')]

## 3. Candidate pool and queries

**What this does.** For each molecule we need a list of suspects: every known structure whose exact mass matches the measured neutral mass. The list of "every known structure" is the union of the training set's 276k structures and **COCONUT**, an open database of about 700k natural products. Structures already in the training set are removed from the COCONUT half so nothing is counted twice.

"Matches" is judged in **ppm** (parts per million). A window of ±10 ppm around a 350 Da molecule is ±0.0035 Da — tiny, and the instrument is accurate enough for it. Within that window a molecule typically has **50–150 suspects**: same mass, different arrangement of atoms. Telling those apart is the whole problem.

The **queries** are the held-out molecules' own spectra. A molecule usually has several (different collision energies, sometimes both polarities); all of them are used together.

> **Expert note.** The pool is stored as arrays sorted by mass so a window lookup is two `searchsorted` calls. COCONUT masses are computed from formula, not taken from the file, so both halves of the pool use one mass convention. The printed spectra-per-molecule figure matters: the public test averages 3.03 spectra per molecule and effect sizes measured at higher counts do not transfer one-to-one.

In [ ]:
co = pd.read_parquet(COCO)
def pick(*names):
    for n in names:
        for c in co.columns:
            if c.lower() == n: return c
c_smi, c_ik, c_form = pick('canonical_smiles', 'smiles'), pick('inchikey', 'inchikey14'), pick('molecular_formula', 'formula')
co = co[[c_smi, c_ik, c_form]].rename(columns={c_smi: 'smi', c_ik: 'ik', c_form: 'formula'})
u = pd.Series(co.formula.astype(str).unique()); co['mass'] = co.formula.astype(str).map(dict(zip(u, u.map(fmass))))
co['k'] = co.ik.astype(str).str[:14]
co = co.dropna(subset=['mass', 'smi']).drop_duplicates('k')
co = co[~co.k.isin(set(s_key))].sort_values('mass').reset_index(drop=True)
U_key = np.concatenate([s_key, co.k.to_numpy()])
U_smiles = np.concatenate([s_smiles, co.smi.astype(str).to_numpy()])
U_mass = np.concatenate([s_mass, co.mass.to_numpy()])
um = np.argsort(U_mass); um = um[np.isfinite(U_mass[um])]; um_sorted = U_mass[um]
def pool(M, ppm=10.0):
    d = max(M * ppm / 1e6, 0.001)
    return um[np.searchsorted(um_sorted, M - d):np.searchsorted(um_sorted, M + d, side='right')]
tick(f'pool: {len(um):,} structures ({len(co):,} COCONUT-only)')
del co; gc.collect()

vq = np.nonzero(IS_NP_EX & tr.adduct.isin(list(ADDUCT_SHIFT)).to_numpy())[0]
VQ = pd.DataFrame({'row': vq, 'mol': ik[vq], 'mode': mode_arr[vq],
                   'M': neutral_mass(prec_arr[vq], tr.adduct.astype(str).to_numpy()[vq])})
print(f'queries: {VQ.mol.nunique()} molecules / {len(VQ)} spectra '
      f'({len(VQ)/VQ.mol.nunique():.2f} spectra per molecule; the public test averages 3.03)')

## 4. Spectral kernels

**What this does.** A spectrum is a list of peaks: an m/z value (the mass of a fragment) and an intensity (how much of it was seen). Comparing two spectra means matching peaks that sit at the same m/z and asking how similar the two intensity patterns are.

Two pieces of code do this:

* **`clean_all`** — the housekeeping every pipeline applies before comparing anything. Test spectra carry hundreds of tiny noise peaks (the median has 230 peaks, but only ~18 are above 1% of the tallest). Cleaning drops peaks below a relative-intensity floor, drops anything heavier than the precursor (a fragment cannot outweigh the ion it came from), keeps the *N* strongest, and rescales so the tallest peak is 1.
* **`pair_sim`** — the similarity itself. We use **spectral entropy similarity**, which treats the intensities as a probability distribution and asks how much information is lost when the two spectra are merged. It is more robust than plain cosine, which lets one or two dominant peaks decide the answer. Peaks are matched **one-to-one, best pairs first**, so a noisy spectrum cannot match the same reference peak repeatedly.

> **Expert note.** `_ent_w` applies the Li et al. entropy weighting (intensities raised to `0.25 + 0.25·H` when spectral entropy `H < 3`) before comparison, which flattens spiky spectra. Matching tolerance is 0.01 Da. Everything is Numba-compiled; `score_pairs` parallelises over (query, reference) pairs with `prange`.

In [ ]:
@njit(cache=False)
def clean_all(mz, it, off, prec, floor, topk, rm_prec):
    n = len(off) - 1
    out_mz = np.empty(len(mz), np.float32); out_it = np.empty(len(mz), np.float32); out_off = np.zeros(n + 1, np.int64)
    w = 0
    for s in range(n):
        a, b = off[s], off[s + 1]; out_off[s] = w
        if b <= a: continue
        p = prec[s]; lim = 1e9
        if np.isfinite(p): lim = p - 1.5 if rm_prec else p + 2.0
        mx = 0.0
        for j in range(a, b):
            if mz[j] <= lim and it[j] > mx: mx = it[j]
        if mx <= 0: continue
        keep = np.empty(b - a, np.int64); c = 0
        for j in range(a, b):
            if mz[j] <= lim and it[j] >= floor * mx: keep[c] = j; c += 1
        keep = keep[:c]
        if c > topk:
            order = np.argsort(-it[keep]); keep = keep[order[:topk]]
        keep = keep[np.argsort(mz[keep])]
        for j in keep:
            out_mz[w] = mz[j]; out_it[w] = it[j] / mx; w += 1
    out_off[n] = w
    return out_mz[:w], out_it[:w], out_off

@njit(cache=False)
def _ent_w(p):
    p = p / p.sum(); H = 0.0
    for v in p:
        if v > 0: H -= v * np.log(v)
    if H < 3.0:
        p = p ** (0.25 + 0.25 * H); p = p / p.sum()
    return p

@njit(cache=False)
def pair_sim(m1, i1, m2, i2, tol):
    n1, n2 = len(m1), len(m2)
    if n1 == 0 or n2 == 0: return 0.0
    a = _ent_w(i1.astype(np.float64)); b = _ent_w(i2.astype(np.float64))
    ps = np.empty(n1 * n2); pi = np.empty(n1 * n2, np.int64); pj = np.empty(n1 * n2, np.int64)
    k = 0; start = 0; ln4 = np.log(4.0)
    for x in range(n1):
        while start < n2 and m2[start] < m1[x] - tol: start += 1
        y = start
        while y < n2 and m2[y] <= m1[x] + tol:
            s = a[x] + b[y]; ps[k] = (s * np.log(s) - a[x] * np.log(a[x]) - b[y] * np.log(b[y])) / ln4
            pi[k] = x; pj[k] = y; k += 1; y += 1
    if k == 0: return 0.0
    order = np.argsort(-ps[:k]); u1 = np.zeros(n1, np.bool_); u2 = np.zeros(n2, np.bool_); tot = 0.0
    for o in order:
        if not u1[pi[o]] and not u2[pj[o]]:
            tot += ps[o]; u1[pi[o]] = True; u2[pj[o]] = True
    return tot

@njit(parallel=True, cache=False)
def score_pairs(qs, rs, cmz, cit, coff, tol):
    out = np.zeros(len(qs))
    for t in prange(len(qs)):
        q, r = qs[t], rs[t]
        out[t] = pair_sim(cmz[coff[q]:coff[q + 1]], cit[coff[q]:coff[q + 1]], cmz[coff[r]:coff[r + 1]], cit[coff[r]:coff[r + 1]], tol)
    return out

### Load the peaks we need, in one streaming pass

**What this does.** `train.parquet` is 3 GB, almost all of it the two peak-array columns. We never need every spectrum's peaks at once — only the analog index (every spectrum outside `enveda-180`, which is drug-like screening chemistry and a poor source of natural-product relatives), the library spectra whose structures sit near a query mass, and the queries themselves. A single pass over the file pulls exactly those rows.

Two cleaned copies are kept: one for library search (0.5% floor, 200 peaks, precursor kept) and one for analog vectors (1% floor, 60 peaks, precursor removed — the precursor is at a different mass for every relative, so it would only add noise to a cross-mass comparison).

> **Expert note.** Arrow `take` on list columns with `list_flatten`/`list_value_length` avoids materialising Python lists. `pos` maps a training row to its slot in the compact peak store; `-1` means "not loaded".

In [ ]:
need = ~IS_E180                                   # analog index: every non-enveda-180 spectrum
need[vq] = True
for M in np.unique(np.round(VQ.M.to_numpy(), 4)):
    for s in lib_structs(M, 0.012):
        need[row_order[s_start[s]:s_end[s]]] = True
pos = np.full(N, -1, np.int64); mzp, itp, lnp = [], [], []
off = k = 0
for batch in pf.iter_batches(batch_size=200_000, columns=['ms2_mzs', 'ms2_normalized_intensities']):
    m = batch.num_rows; sel = np.nonzero(need[off:off + m])[0]
    if len(sel):
        idx = pa.array(sel); a = batch.column(0).take(idx); b = batch.column(1).take(idx)
        lnp.append(pc.list_value_length(a).fill_null(0).to_numpy(zero_copy_only=False))
        mzp.append(pc.list_flatten(a).to_numpy(zero_copy_only=False).astype(np.float32))
        itp.append(pc.list_flatten(b).to_numpy(zero_copy_only=False).astype(np.float32))
        pos[off + sel] = np.arange(k, k + len(sel)); k += len(sel)
    off += m
RAW_MZ = np.concatenate(mzp); RAW_IT = np.nan_to_num(np.concatenate(itp))
RAW_OFF = np.concatenate([[0], np.cumsum(np.concatenate(lnp))]).astype(np.int64)
del mzp, itp, lnp; gc.collect()
srow = np.full(k, -1, np.int64); srow[pos[pos >= 0]] = np.nonzero(pos >= 0)[0]
PREC = prec_arr[srow]
C_MZ, C_IT, C_OFF = clean_all(RAW_MZ, RAW_IT, RAW_OFF, PREC, 0.005, 200, False)   # library search
A_MZ, A_IT, A_OFF = clean_all(RAW_MZ, RAW_IT, RAW_OFF, PREC, 0.01, 60, True)      # analog vectors
del RAW_MZ, RAW_IT; gc.collect()
tick(f'peak store: {k:,} spectra')

### Verify the holdout: the library search must find nothing

**What this does.** This is the proof that Section 2 worked. For every query spectrum we run the ordinary library search — compare it against every library spectrum of every structure within 0.01 Da — and check whether it ever retrieves *its own* structure. In a correct Class-2 simulation that must never happen, because all of the structure's spectra were removed.

The cell prints the number of self-matches and stops the notebook if it is not zero. Everything after this point can be trusted only because this check passes.

In [ ]:
qs, rs, ms, ss = [], [], [], []
for M, mol, row in zip(VQ.M.to_numpy(), VQ.mol.to_numpy(), VQ.row.to_numpy()):
    st = lib_structs(M, 0.01)
    if not len(st): continue
    rows = np.concatenate([row_order[s_start[s]:s_end[s]] for s in st])
    rows = rows[LIB_OK[rows] & (pos[rows] >= 0)]
    if not len(rows): continue
    rs.append(pos[rows]); ss.append(ik[rows]); qs.append(np.full(len(rows), pos[row], np.int64)); ms.append(np.full(len(rows), mol))
sims = score_pairs(np.concatenate(qs), np.concatenate(rs), C_MZ, C_IT, C_OFF, 0.01)
LIBSIM = pd.DataFrame({'mol': np.concatenate(ms), 's': np.concatenate(ss), 'sim': sims}).groupby(['mol', 's'], sort=False).sim.max().reset_index()
self_hits = LIBSIM[LIBSIM.mol == LIBSIM.s]
print(f'library comparisons: {len(sims):,}')
print(f'exact self-matches after removal: {len(self_hits)}  (best self-similarity {self_hits.sim.max() if len(self_hits) else 0.0:.3f})')
assert len(self_hits) == 0, 'holdout leaks - every number below would be inflated'

## 5. Rank the candidates (analog propagation)

**What this does.** A Class-2 molecule has no spectrum of its own in the library — but its chemical **relatives** usually do. Two molecules that share a skeleton and differ by one small group (an extra oxygen, an extra methyl, a sugar) break into largely the same pieces. So the idea, in three steps:

1. Find library spectra that **look like the query at any mass** (not just the same mass).
2. Look up what molecules those spectra belong to — these are the query's likely relatives.
3. Prefer suspects whose *structure* resembles those relatives.

Step 1 needs a fast way to compare a spectrum against a million others. Each spectrum becomes a sparse vector over 0.1 Da bins in two channels: the **fragment masses** themselves, and the **neutral losses** (precursor mass minus fragment mass — the mass of the piece that fell off). The loss channel is what lets a molecule and its hydroxylated cousin match even though every fragment m/z differs: the *losses* are shared. The search is then one sparse matrix product.

Step 3 uses **molecular fingerprints** — a molecule turned into a long binary vector of "contains this substructure" flags — and **Tanimoto similarity** (shared bits ÷ total bits) between each suspect and each relative, weighted by how strongly that relative's spectrum matched.

This ranking is the standard Class-2 method in the strong public pipelines. It is not the channel under test — it is the thing that produces the **top-3 list** we will then interrogate.

> **Expert note.** Fingerprint = ECFP4 ‖ ECFP6 (2 × 2048 bits). Relatives are limited to the same ionisation polarity as the query. Analog weight is `similarity³`, so the closest relatives dominate; score = max over relatives of `sim³ · Tanimoto`. The public pipelines add a learned ranker on top of this; we deliberately use the bare analog score so the top-3 does not already depend on the fragmentation channel we are about to test.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator
FPG2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
FPG3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
_FP = {}
def fps(us):
    out = np.zeros((len(us), 4096), np.float32)
    for i, u in enumerate(us):
        u = int(u)
        if u not in _FP:
            m = Chem.MolFromSmiles(U_smiles[u]) if isinstance(U_smiles[u], str) else None
            _FP[u] = np.concatenate([FPG2.GetFingerprintAsNumPy(m), FPG3.GetFingerprintAsNumPy(m)]).astype(np.uint8) if m is not None else None
        if _FP[u] is not None: out[i] = _FP[u]
    return out
def tanimoto(A, B):
    inter = A @ B.T; na = A.sum(1)[:, None]; nb = B.sum(1)[None, :]
    return np.where(na + nb - inter > 0, inter / np.maximum(na + nb - inter, 1), 0.0)

BIN, NB_MZ, NB_NL = 0.1, 10000, 5000
def vectors(store_idx):
    st = A_OFF[store_idx]; ln = A_OFF[store_idx + 1] - st
    rowid = np.repeat(np.arange(len(store_idx)), ln)
    pk = np.repeat(st - np.concatenate([[0], np.cumsum(ln)[:-1]]), ln) + np.arange(ln.sum())
    mz = A_MZ[pk].astype(np.float64); w = np.sqrt(A_IT[pk]).astype(np.float32); prec = PREC[store_idx][rowid]
    b1 = np.floor(mz / BIN).astype(np.int64); m1 = (b1 >= 0) & (b1 < NB_MZ)
    nl = prec - mz; b2 = np.floor(np.nan_to_num(nl, nan=-1) / BIN).astype(np.int64); m2 = np.isfinite(nl) & (b2 >= 1) & (b2 < NB_NL)
    X = sp.csr_matrix((np.concatenate([w[m1], 0.5 * w[m2]]), (np.concatenate([rowid[m1], rowid[m2]]),
                       np.concatenate([b1[m1], NB_MZ + b2[m2]]))), shape=(len(store_idx), NB_MZ + NB_NL))
    X.sum_duplicates()
    nrm = np.sqrt(np.asarray(X.multiply(X).sum(1)).ravel()); nrm[nrm == 0] = 1
    return (sp.diags(1 / nrm) @ X).tocsr().astype(np.float32)

LIBA = np.nonzero(~IS_E180 & (pos >= 0) & LIB_OK)[0]         # analog references, Class-2 clean
BYMODE = {m: LIBA[mode_arr[LIBA] == m] for m in (0, 1)}
LIBX = {m: vectors(pos[BYMODE[m]]) for m in (0, 1)}
tick(f'analog index: {LIBX[1].shape[0]:,} pos / {LIBX[0].shape[0]:,} neg spectra')

def analog_search(qidx, qmode, K=200):
    R = np.full((len(qidx), K), -1, np.int64); V = np.zeros((len(qidx), K), np.float32)
    for m in (0, 1):
        qi = np.nonzero(qmode == m)[0]
        if not len(qi): continue
        Qd = vectors(qidx[qi]).toarray().T
        X, rows = LIBX[m], BYMODE[m]
        for c in range(0, len(qi), 48):
            Sm = np.asarray(X @ Qd[:, c:c + 48])
            top = np.argpartition(-Sm, K, axis=0)[:K]
            for j in range(top.shape[1]):
                t = top[:, j]; o = np.argsort(-Sm[t, j]); t = t[o]
                R[qi[c + j]] = rows[t]; V[qi[c + j]] = Sm[t, j]
    return R, V

## 6. MetFrag-lite, and the variants we will compare

**What this does.** This is the channel under test. The idea is appealing: instead of comparing the query to *other* spectra, judge each suspect **on its own structure**. Run the experiment in reverse on paper — break the suspect's bonds, weigh the pieces, and ask how much of the observed spectrum those pieces could explain. A suspect whose pieces account for 85% of the signal looks more plausible than one that accounts for 30%.

**A tiny worked example.** Take butane, a chain of four carbons (`C-C-C-C`, mass 58.08 with hydrogens). Breaking one bond gives pieces of 1, 2 or 3 carbons (masses ≈ 15, 29, 43); breaking two bonds can only give the same pieces again. So butane's fragment set is {15, 29, 43, 58}. If the observed spectrum has peaks at 29 and 43 (plus a proton each), butane "explains" them; a peak at 500 it cannot.

Concretely, for each suspect the code:
1. builds the molecular graph (atoms with their hydrogens as weights, bonds as edges),
2. removes every single bond and every pair of bonds, and records the mass of each connected piece,
3. for each observed peak, checks whether some piece (allowing the charge carrier and up to ±*nh* hydrogen atoms to move during fragmentation) lands within *tol* Da of it,
4. reports the share of total (square-rooted) intensity that was explained.

**The two extra quantities.** Alongside the standard score we compute two things the public pipelines do not:

* **`nfrag`** — how many *distinct* fragment masses the suspect can produce at all. Two molecules with the same formula can differ here: every **ring** needs *two* bond breaks to open, so a ring-rich structure yields fewer distinct pieces than a chain-like one of the same mass. We include it to test a hypothesis — that the explain-score mostly measures this complexity rather than chemistry. (Section 8 reports whether the hypothesis survived. It did not.)
* **chance-corrected scores** — a suspect with 300 fragment masses covers a large share of the m/z axis and will "explain" peaks by luck; one with 40 will not. `coverage` estimates that luck (fragment count × matching window ÷ mass range), and the corrected scores subtract it (`excess`) or divide by it (`ratio`).

> **Expert note.** Pieces are found with a union-find over the bond list with the removed bonds skipped; pairs are capped at 1,500 for speed. Masses are deduplicated to 4 decimals. `explain` does a binary search for each peak over `frag ± d·H + charge`, `d ∈ [−nh, nh]`. Coverage assumes non-overlapping windows, which overestimates slightly for large fragment sets — conservative in the direction that matters.

In [ ]:
FRAG_H = 1.00782503207; FRAG_PROTON = FRAG_H - 0.000548579909
def mol_graph(smi):
    m = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
    if m is None or m.GetNumAtoms() > 120: return None
    w = np.zeros(m.GetNumAtoms())
    for a in m.GetAtoms():
        e = MONO.get(a.GetSymbol())
        if e is None: return None
        w[a.GetIdx()] = e + a.GetTotalNumHs() * FRAG_H
    bonds = np.array([[b.GetBeginAtomIdx(), b.GetEndAtomIdx()] for b in m.GetBonds()], np.int64)
    if len(bonds) == 0 or len(bonds) > 60: return None
    return w, bonds

@njit(cache=False)
def _pieces(w, bonds, da, db):
    n = len(w); parent = np.arange(n)
    for bi in range(len(bonds)):
        if bi == da or bi == db: continue
        x, y = bonds[bi, 0], bonds[bi, 1]
        while parent[x] != x: x = parent[x]
        while parent[y] != y: y = parent[y]
        if x != y: parent[x] = y
    tot = np.zeros(n)
    for i in range(n):
        r = i
        while parent[r] != r: r = parent[r]
        tot[r] += w[i]
    out = np.empty(n); c = 0
    for i in range(n):
        if tot[i] > 0: out[c] = tot[i]; c += 1
    return out[:c]

@njit(cache=False)
def fragment_masses(w, bonds, max_pairs):
    nb = len(bonds); buf = np.empty((nb + max_pairs + 1) * 8); c = 0
    for m in _pieces(w, bonds, -1, -1): buf[c] = m; c += 1
    for i in range(nb):
        for m in _pieces(w, bonds, i, -1):
            buf[c] = m; c += 1
    done = 0
    for i in range(nb):
        for j in range(i + 1, nb):
            if done >= max_pairs: break
            for m in _pieces(w, bonds, i, j):
                if c < len(buf): buf[c] = m; c += 1
            done += 1
        if done >= max_pairs: break
    return np.unique(np.round(buf[:c], 4))

@njit(cache=False)
def explain(frag, mz, inten, tol, positive, nh):
    if len(mz) == 0 or len(frag) == 0: return 0.0
    tot = 0.0; hit = 0.0; shift = FRAG_PROTON if positive else -FRAG_PROTON
    for p in range(len(mz)):
        w = np.sqrt(inten[p]); tot += w
        target = mz[p] - shift; ok = False
        for d in range(-nh, nh + 1):
            t = target - d * FRAG_H
            lo, hi = 0, len(frag) - 1
            while lo <= hi:
                mid = (lo + hi) // 2
                if frag[mid] < t - tol: lo = mid + 1
                elif frag[mid] > t + tol: hi = mid - 1
                else: ok = True; break
            if ok: break
        if ok: hit += w
    return hit / tot if tot > 0 else 0.0

_FR = {}
def frag_of(u):
    u = int(u)
    if u not in _FR:
        g = mol_graph(U_smiles[u]); _FR[u] = None if g is None else fragment_masses(g[0], g[1], 1500)
    return _FR[u]
def coverage(fm, prec, tol, nh):
    if fm is None or len(fm) == 0 or not np.isfinite(prec) or prec <= 0: return 1.0
    return float(min(1.0, len(fm) * (2 * nh + 1) * 2 * tol / max(prec, 1.0)))

## 7. The measurement

**What this does, and why it is framed this way.** The competition metric, **MRR@25**, gives full credit for the truth at rank 1, half credit at rank 2, a third at rank 3, and so on. The largest possible gain therefore comes from molecules where the truth is *almost* first — sitting at rank 2 or 3 behind a wrong same-mass isomer. If a channel can pick the truth out of those top-3 lists, it is worth a lot; if it cannot, it is worth little no matter how it looks in other tests.

So for every held-out molecule the code:
1. ranks its suspects by analog propagation (Section 5) and takes the **top 3**;
2. keeps only the **fixable failures** — cases where the truth is in that three but not first;
3. asks each fragmentation variant: *which of these three do you think is the truth?*
4. counts how often it is right.

**How to read the result.** With three options, guessing is right 1 time in 3, so **0.333 is the line a useful signal must beat**. Because the number of fixable cases is modest, the table also prints one **standard error** — the typical wobble you would expect from chance alone. A rate more than about two standard errors below 0.333 is not bad luck; it means the score is systematically pointing at the wrong isomer.

> **Expert note.** The top-3 comes from the bare analog score, so the fragmentation variants are evaluated as *independent* evidence, not as a re-weighting of a ranker that already consumed them. Ties in `argmax` go to the lower index (the higher-ranked candidate), which if anything favours the incumbent, not the truth.

In [ ]:
R, V = analog_search(np.array([pos[r] for r in VQ.row]), VQ['mode'].to_numpy())
VARIANTS = ['nh0_tol0.005', 'nh1_tol0.01', 'nh2_tol0.01 (as used)', 'nfrag (fewer is better)',
            'excess = explain - coverage', 'ratio  = explain / coverage']
picks = {v: 0 for v in VARIANTS}; cases = 0; rank_hist = []
for mol, idx in VQ.groupby('mol', sort=False).indices.items():
    Ms = VQ.M.to_numpy()[idx]
    cand = np.unique(np.concatenate([pool(M) for M in Ms]))
    if not len(cand) or not (U_key[cand] == s_key[mol]).any(): continue
    rr, vv = R[idx].ravel(), V[idx].ravel(); ok = (rr >= 0) & LIB_OK[np.maximum(rr, 0)]
    an = pd.Series(vv[ok]).groupby(ik[rr[ok]]).max().sort_values(ascending=False).head(50)
    if not len(an): continue
    FC = fps(cand); FA = fps(an.index.to_numpy()); aw = an.to_numpy() ** 3.0
    score = (tanimoto(FC, FA) * aw[None, :]).max(1)                      # analog propagation ranking
    order = np.argsort(-score, kind='stable'); truth = U_key[cand] == s_key[mol]
    hit = np.nonzero(truth[order])[0]; rank_hist.append(hit[0] + 1 if len(hit) else 10**6)
    top = order[:3]
    if not truth[top].any() or truth[top[0]]: continue                   # only the fixable failures
    cases += 1; ti = int(np.nonzero(truth[top])[0][0])
    specs = [(C_MZ[C_OFF[pos[r]]:C_OFF[pos[r] + 1]].astype(np.float64),
              C_IT[C_OFF[pos[r]]:C_OFF[pos[r] + 1]].astype(np.float64), bool(mode_arr[r])) for r in VQ.row.to_numpy()[idx]]
    specs = [s for s in specs if len(s[0])]
    if not specs: cases -= 1; continue
    prec0 = float(np.median([s[0].max() for s in specs]))
    sc = {v: np.zeros(3) for v in VARIANTS}
    for j, u in enumerate(top):
        fm = frag_of(cand[u]); n_fr = 0 if fm is None else len(fm)
        e0 = max((explain(fm, a, b, 0.005, p, 0) for a, b, p in specs), default=0.0) if fm is not None else 0.0
        e1 = max((explain(fm, a, b, 0.01, p, 1) for a, b, p in specs), default=0.0) if fm is not None else 0.0
        e2 = max((explain(fm, a, b, 0.01, p, 2) for a, b, p in specs), default=0.0) if fm is not None else 0.0
        cov = coverage(fm, prec0, 0.01, 2)
        sc['nh0_tol0.005'][j] = e0; sc['nh1_tol0.01'][j] = e1; sc['nh2_tol0.01 (as used)'][j] = e2
        sc['nfrag (fewer is better)'][j] = -n_fr
        sc['excess = explain - coverage'][j] = e2 - cov
        sc['ratio  = explain / coverage'][j] = e2 / max(cov, 1e-3)
    for v in VARIANTS:
        if int(np.argmax(sc[v])) == ti: picks[v] += 1
tick(f'measured on {cases} fixable cases')

res = pd.DataFrame([{'variant': v, 'picked the truth': picks[v], 'of cases': cases, 'rate': picks[v] / max(cases, 1)} for v in VARIANTS])
res = res.sort_values('rate')
se = (0.333 * 0.667 / max(cases, 1)) ** 0.5
display(res.round(3).style.hide(axis='index'))
note(f'Random choice among 3 gives **0.333**; with {cases} cases one standard error is **{se:.3f}**.')
print(res.round(3).to_string(index=False)); print(f'random = 0.333, one standard error = {se:.3f}, cases = {cases}')
res.assign(cases=cases, random=1/3, std_err=se).to_csv('fragmentation_top3_results.csv', index=False)
pd.Series(rank_hist, name='truth_rank').to_csv('truth_rank_cold.csv', index=False)

### The picture

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.4), gridspec_kw={'width_ratios': [1.35, 1]})
cols = [ORANGE if r < 0.333 - se else (AQUA if r > 0.333 + se else MUTED) for r in res.rate]
ax[0].barh(res.variant, res.rate, color=cols, height=0.6)
ax[0].axvline(0.333, color=INK2, lw=2); ax[0].text(0.336, -0.45, 'random', color=INK2, fontsize=9)
ax[0].axvspan(0.333 - se, 0.333 + se, color=GRID, alpha=0.6, zorder=0)
ax[0].set(title='Does the fragmentation score pick the true structure out of the top 3?', xlabel='share of cases', xlim=(0, 0.5))
ax[0].grid(axis='y', visible=False)
for y, r in enumerate(res.rate): ax[0].text(r + 0.006, y, f'{r:.3f}', va='center', fontsize=9, color=INK2)
rh = pd.Series(rank_hist); vc = pd.cut(rh, [0, 1, 2, 3, 5, 10, 25, 10**7], labels=['1', '2', '3', '4-5', '6-10', '11-25', '>25']).value_counts().reindex(['1', '2', '3', '4-5', '6-10', '11-25', '>25'])
ax[1].bar(vc.index.astype(str), vc.values, color=[AQUA] + [BLUE] * 2 + [MUTED] * 4, width=0.65)
ax[1].set(title='Where the true structure ranks (Class-2 simulation)'); ax[1].grid(axis='x', visible=False)
for x, v in enumerate(vc.values): ax[1].text(x, v + 0.5, str(int(v)), ha='center', fontsize=9, color=INK2)
plt.tight_layout(); plt.show()

## 8. What this means

**1. On close isomers, the common setting does not add discrimination.** The ±2 hydrogen, 0.01 Da explain-score picks the truth in 22% of fixable cases against 33% for guessing, about two standard errors below chance on this sample. Used as positive evidence in a ranker, that would tend to move the correct structure *down* among its plausible isomers — worth checking with an ablation in your own pipeline.

**2. No variant rises above chance here.** Stricter matching (fewer hydrogen rearrangements, tighter tolerance) and correcting for the m/z coverage a candidate gets for free both move the rate *toward* 33%, which suggests part of what the raw score captures is candidate complexity rather than fit. The simplest reading is that combinatorial bond-breaking, as implemented, does not separate same-formula isomers: their reachable fragment masses overlap too much.

**3. A hypothesis that did not survive.** In an earlier version of this measurement, built on a different ranking, "prefer the candidate with fewer fragment masses" scored as well as the corrected variants, which suggested the channel was mostly measuring molecular complexity (rings need two breaks, so ring-rich natural products yield fewer masses). In this leak-checked setup `nfrag` is the *worst* variant. That explanation is not supported, and it is left in the table as a negative result rather than a claim.

**Practical takeaways**

* Judge a candidate-scoring channel on the **top-3 choice**, not on standalone MRR. A channel can look useful in an ablation — by separating plausible candidates from implausible ones, which the mass window and analog channel already do — and still be neutral or harmful where the remaining score lives.
* If a ranker already consumes a MetFrag-lite score, it is worth an ablation *without* it on a properly held-out Class-2 set. It may be free to drop.
* A fragmentation channel that could separate isomers has to model *which* bonds break — bond energies, charge retention, rearrangements — not merely which masses are reachable.

**Caveats.** One simulation (250 held-out natural-product structures, timsTOF), one ranking (analog propagation), and the sample of fixable cases is small — the standard error is printed above. The holdout is verified free of self-matches, which is the failure mode that would inflate all of this. Corrections and contradicting runs are very welcome.

### Reusing this harness

The notebook is a general test bench for **any** candidate-scoring idea, not just fragmentation: compute your score for each of the three candidates inside the loop in Section 7, add it to `VARIANTS`, and the table and chart will include it. The Class-2 simulation, the leak check and the top-3 framing are the parts worth keeping.

---

## Glossary

| Term | Meaning |
|---|---|
| **MS/MS spectrum** | A list of (m/z, intensity) peaks recorded after a molecule is ionised, isolated and smashed into fragments |
| **m/z** | Mass-to-charge ratio of an ion; for singly charged ions, effectively its mass |
| **Precursor** | The intact ion selected for fragmentation; `precursor_mz` is its m/z |
| **Adduct** | The charged particle attached to the molecule during ionisation, e.g. `[M+H]+` = molecule plus a proton |
| **Neutral mass** | Mass of the bare molecule, obtained by undoing the adduct |
| **Monoisotopic mass** | Mass computed from the most common isotope of each atom |
| **ppm** | Parts per million; ±10 ppm at 350 Da is ±0.0035 Da |
| **Candidate / suspect** | A known structure whose mass matches the query within the ppm window |
| **Isomers** | Molecules with the same formula (hence the same mass) but different atom arrangements |
| **InChIKey14** | First 14 characters of a standard chemical identifier; encodes connectivity, ignores stereochemistry; the competition's notion of "same molecule" |
| **Class 1 / 2 / 3** | Test molecule has reference spectra / has a known structure but no spectra / is entirely novel |
| **MRR@25** | Mean reciprocal rank: 1 for truth at rank 1, ½ at rank 2, ⅓ at rank 3 … 0 beyond 25 |
| **Entropy similarity** | A spectral similarity that treats intensities as a probability distribution; robust to a few dominant peaks |
| **Neutral loss** | Precursor mass minus fragment mass: the mass of the piece that fell off |
| **Analog propagation** | Find spectra similar to the query at any mass, then prefer candidates that look like those spectra's molecules |
| **Fingerprint / Tanimoto** | A molecule as a binary "contains substructure X" vector, and the overlap between two such vectors |
| **MetFrag-lite** | In-silico fragmentation: break bonds on paper and see how much of the spectrum the pieces explain |
| **`nfrag`** | Number of distinct fragment masses a candidate can produce |
| **Coverage** | Estimated share of the m/z axis a candidate's fragment set covers — its chance of explaining a random peak |
| **Fixable failure** | A molecule whose truth is in the top 3 but not at rank 1 |
| **Standard error** | Typical random wobble of a measured rate given the sample size |